# Titanic Survival Prediction — Without a Pipeline

A full manual preprocessing workflow: impute missing values, one-hot encode categoricals, train a model, and save every piece needed to reuse it later. This notebook exists as a deliberate contrast to `titanic-using-pipeline.ipynb` — same end result, but tracking every transformer object by hand instead of bundling them into one `Pipeline`.

## 1. Setup and Cleaning

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

`PassengerId`, `Name`, `Ticket`, and `Cabin` are dropped — `PassengerId` carries no information, `Name`/`Ticket` need heavy feature engineering to be useful (not covered here), and `Cabin` is missing for the large majority of passengers.

In [ ]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [ ]:
df.head()

In [ ]:
# Step 1 -> train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

Splitting **before** any imputing or encoding — every transformer below must learn its parameters from `X_train` only, per the same principle covered in the earlier scaling/encoding notebooks.

In [ ]:
X_train.head(2)

In [ ]:
y_train.head()

In [ ]:
df.isnull().sum()

`Age` and `Embarked` are the only columns with missing values — confirming what needs imputing before anything else can proceed.

## 2. Imputing Missing Values

Two different imputers, since a numeric column and a categorical column need different fill strategies: `SimpleImputer()` defaults to `strategy='mean'` (right for `Age`), while `Embarked` — a category, not a number — needs `strategy='most_frequent'` (fill with the most common port).

In [ ]:
# Applying imputation

si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

In [ ]:
X_train_embarked

## 3. One-Hot Encoding `Sex` and `Embarked`

Both are nominal categories with no natural order, so `OneHotEncoder` is the right tool — same reasoning as the earlier encoding notebook.

### ⚠️ Two Issues in This Cell

1. **Deprecated argument:** `sparse=False` was removed in scikit-learn 1.4 — renamed to `sparse_output=False`.
2. **Missing step:** the last line, `X_test_embarked` on its own, does nothing — it looks up the variable and discards the result. The clear intent was `X_test_embarked = ohe_embarked.transform(X_test_embarked)`, encoding the test set's (already-imputed) `Embarked` values with the *same* encoder fitted on `X_train`. Without this line, `X_test_embarked` is left holding the raw **imputed-but-not-yet-encoded** text values — concatenating that together with the one-hot-encoded `X_train_embarked` later would produce mismatched, inconsistent columns between train and test.

In [ ]:
# one hot encoding Sex and Embarked

ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test_embarked)   # fixed: this was missing


In [ ]:
X_train_embarked

In [ ]:
X_train.head(2)

## 4. Reassembling Everything

Dropping the columns that were just individually transformed leaves `Pclass`, `SibSp`, `Parch`, `Fare` — the columns that didn't need any preprocessing, kept exactly as they were.

In [ ]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])

In [ ]:
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

In [ ]:
X_train_transformed = np.concatenate((X_train_rem,X_train_age,X_train_sex,X_train_embarked),axis=1)
X_test_transformed = np.concatenate((X_test_rem,X_test_age,X_test_sex,X_test_embarked),axis=1)

In [ ]:
X_test_transformed.shape

## 5. Training and Evaluating

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed,y_train)

In [ ]:
y_pred = clf.predict(X_test_transformed)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

### A More Reliable Estimate — Cross-Validation

A single train/test split's accuracy can vary a fair bit depending on which rows happened to land in the test set. `cross_val_score` fits and evaluates the model several times on different splits and averages the result — a more trustworthy estimate of real-world performance (same technique used in the binarization/discretization notebooks).

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

X_full_transformed = np.concatenate((
    df.drop(columns=['Survived', 'Sex', 'Age', 'Embarked']),
    si_age.fit_transform(df[['Age']]),
    ohe_sex.fit_transform(df[['Sex']]),
    ohe_embarked.fit_transform(si_embarked.fit_transform(df[['Embarked']]))
), axis=1)

np.mean(cross_val_score(DecisionTreeClassifier(), X_full_transformed, df['Survived'], cv=10, scoring='accuracy'))


## 6. Saving Every Piece for Later Use

This is the part where the "without pipeline" approach shows its real cost: **every single transformer** used has to be saved separately, and whoever loads them later must remember the exact order and method to reapply each one — exactly as done manually above.

In [ ]:
import pickle

In [ ]:
pickle.dump(ohe_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf,open('models/clf.pkl','wb'))

### ⚠️ A Real Gap: `si_age` and `si_embarked` Are Never Saved

Only `ohe_sex`, `ohe_embarked`, and `clf` get pickled — the two `SimpleImputer`s are not. That's fine *only* if every future prediction request is guaranteed to arrive with `Age` and `Embarked` already filled in (as `predict-without-pipeline.ipynb` happens to assume). If a real input ever came in with a missing `Age`, there'd be no saved imputer available to fill it — worth saving these too, for completeness.

In [ ]:
pickle.dump(si_age, open('models/si_age.pkl', 'wb'))
pickle.dump(si_embarked, open('models/si_embarked.pkl', 'wb'))


## Summary

| Step | Method |
|---|---|
| Impute numeric column | `SimpleImputer()` *(default: mean)* |
| Impute categorical column | `SimpleImputer(strategy='most_frequent')` |
| One-hot encode | `OneHotEncoder(sparse_output=False, handle_unknown='ignore')` |
| Fit once, transform both | `.fit_transform(X_train)` then `.transform(X_test)` — **never** `fit_transform` on test |
| Combine pieces back together | `np.concatenate((...), axis=1)` |
| More reliable accuracy estimate | `cross_val_score(model, X, y, cv=10)` |
| Persist a model/transformer | `pickle.dump(obj, open('file.pkl', 'wb'))` |

### The core weakness of this approach
Every transformer (`si_age`, `si_embarked`, `ohe_sex`, `ohe_embarked`) is a separate object that must be individually saved, individually loaded, and applied in exactly the right order at prediction time — miss one (as happened here with the imputers) and predictions on new data will silently be wrong or crash. `titanic-using-pipeline.ipynb` solves this by bundling everything into a single `Pipeline` object — one thing to save, one thing to load, impossible to apply out of order.